# 0. Imports

In [1]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor
import lightgbm
import scipy
import statsmodels
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

# 1. Questions


### 1.1 Вывод аналитического решения задачи регрессии. 

Функция потерь для линейной регрессии — это среднеквадратичная ошибка (MSE)
$$L(\mathbf{w}) = \|\mathbf{Xw} - \mathbf{y}\|_2^2 = (\mathbf{Xw} - \mathbf{y})^T(\mathbf{Xw} - \mathbf{y})$$

$$L(\mathbf{w}) = ((\mathbf{Xw})^T - \mathbf{y}^T)(\mathbf{Xw} - \mathbf{y}) = \mathbf{w}^T\mathbf{X}^T\mathbf{Xw} - \mathbf{w}^T\mathbf{X}^T\mathbf{y} - \mathbf{y}^T\mathbf{Xw} + \mathbf{y}^T\mathbf{y}$$

Так как $\mathbf{w}^T\mathbf{X}^T\mathbf{y}$ — это скаляр, он равен своему транспонированному значению $(\mathbf{w}^T\mathbf{X}^T\mathbf{y})^T = \mathbf{y}^T\mathbf{Xw}$. Следовательно:
$$L(\mathbf{w}) = \mathbf{w}^T\mathbf{X}^T\mathbf{Xw} - 2\mathbf{w}^T\mathbf{X}^T\mathbf{y} + \mathbf{y}^T\mathbf{y}$$

Чтобы найти минимум, берем градиент по $\mathbf{w}$ и приравниваем его к нулю:
$$\nabla_\mathbf{w} L(\mathbf{w}) = 2\mathbf{X}^T\mathbf{Xw} - 2\mathbf{X}^T\mathbf{y} = 0$$
$$\mathbf{X}^T\mathbf{Xw} = \mathbf{X}^T\mathbf{y}$$

$$\mathbf{w} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

### 1.2 Что меняется в решении при добавлении регуляризаций L1 и L2 в функцию потерь.

**L2-регуляризация (Ridge-регрессия):**
Функция потерь принимает вид:
$$L(\mathbf{w}) = \|\mathbf{Xw} - \mathbf{y}\|_2^2 + \lambda \|\mathbf{w}\|_2^2$$
Аналитическое решение:
$$\mathbf{w} = (\mathbf{X}^T\mathbf{X} + \lambda \mathbf{I})^{-1}\mathbf{X}^T\mathbf{y}$$


**L1-регуляризация (Lasso-регрессия):**
Функция потерь принимает вид:
$$L(\mathbf{w}) = \|\mathbf{Xw} - \mathbf{y}\|_2^2 + \lambda \|\mathbf{w}\|_1$$
Для Lasso не существует закрытого аналитического решения, так как норма L1 не дифференцируема в нуле. Обычно она решается итерационными методами, такими как градиентный спуск. Lasso приводит к разреженности весов (некоторые веса становятся ровно нулями).

### 1.3 Почему регуляризация L1 часто используется для отбора признаков. Почему после обучения модели многие веса равны 0?

L1-регуляризация добавляет штраф, равный сумме абсолютных значений весов. В пространстве параметров ограничение $\|\mathbf{w}\|_1 \le t$ образует форму ромба. Линии уровня функции потерь MSE представляют собой эллипсы. Оптимальное решение находится в точке, где эллипс впервые касается ромба. Из-за острых углов на осях касание с большой вероятностью происходит именно в вершине, где одна или несколько координат весов равны нулю. Это эффективно «отбирает» признаки, исключая те, чьи веса стали нулевыми.

### 1.4 Как можно использовать те же модели (линейная регрессия, Ridge и т. д.), но сделать возможным обучение нелинейных зависимостей.

Линейные модели являются «линейными» относительно своих параметров $\mathbf{w}$, а не обязательно относительно входных признаков $\mathbf{X}$. Мы можем обучать нелинейные зависимости, преобразуя исходные признаки в пространство более высокой размерности с помощью базисных функций $\phi(\mathbf{x})$. Например, полиномиальная регрессия использует преобразования вида $x \to [1, x, x^2, x^3, \dots, x^n]$. Модель принимает вид:
$$f(\mathbf{x}) = \mathbf{w}^T \phi(\mathbf{x})$$
Хотя $f(\mathbf{x})$ нелинейна относительно $\mathbf{x}$, она остается линейной комбинацией параметров $\mathbf{w}$, поэтому мы можем использовать те же алгоритмы линейной регрессии.

# 2. Preprocessing 

In [2]:
test = pd.read_json('../datasets/test.json')[['bathrooms', 'bedrooms', 'price', 'features']]
train = pd.read_json('../datasets/train.json')[['bathrooms', 'bedrooms', 'price', 'features']]
test.shape, train.shape

((74659, 4), (49352, 4))

# 3. Data analysis part 2

In [3]:
test.columns, train.columns

(Index(['bathrooms', 'bedrooms', 'price', 'features'], dtype='str'),
 Index(['bathrooms', 'bedrooms', 'price', 'features'], dtype='str'))

In [4]:
test.head()

,bathrooms,bedrooms,price,features
0,1.0,1,2950,"[Elevator, Laundry in Building, Laundry in Uni..."
1,1.0,2,2850,"[Pre-War, Dogs Allowed, Cats Allowed]"
2,1.0,0,2295,"[Pre-War, Dogs Allowed, Cats Allowed]"
3,1.0,2,2900,"[Hardwood Floors, Dogs Allowed, Cats Allowed]"
5,1.0,1,3254,"[Roof Deck, Doorman, Elevator, Fitness Center,..."


In [5]:
train.head()

,bathrooms,bedrooms,price,features
4,1.0,1,2400,"[Dining Room, Pre-War, Laundry in Building, Di..."
6,1.0,2,3800,"[Doorman, Elevator, Laundry in Building, Dishw..."
9,1.0,2,3495,"[Doorman, Elevator, Laundry in Building, Laund..."
10,1.5,3,3000,[]
15,1.0,0,2795,"[Doorman, Elevator, Fitness Center, Laundry in..."


In [6]:
train['features'].head(10)

4     [Dining Room, Pre-War, Laundry in Building, Di...
6     [Doorman, Elevator, Laundry in Building, Dishw...
9     [Doorman, Elevator, Laundry in Building, Laund...
10                                                   []
15    [Doorman, Elevator, Fitness Center, Laundry in...
16    [Doorman, Elevator, Loft, Dishwasher, Hardwood...
18    [Fireplace, Laundry in Unit, Dishwasher, Hardw...
19    [Elevator, Laundry in Building, Dishwasher, Ha...
23                                    [Hardwood Floors]
32                         [Cats Allowed, Dogs Allowed]
Name: features, dtype: object

In [7]:
def clear_features(df_iterator):
    all_features = []

    for _, row in df_iterator:
        features_str = str(row["features"])
        features_str = re.sub(r'[\[\]\'\"\s]', '', features_str)

        features = [f.strip() for f in features_str.split(',') if f.strip()]
        all_features.extend(features)
    return all_features

In [8]:
all_features = clear_features(train.iterrows())

In [9]:
unique_features = set(all_features)
len(unique_features)

1546

In [10]:
feature_counter = Counter(all_features)
top_20_features = feature_counter.most_common(20)
top_20_features = [f[0] for f in top_20_features]
top_20_features

['Elevator',
 'CatsAllowed',
 'HardwoodFloors',
 'DogsAllowed',
 'Doorman',
 'Dishwasher',
 'NoFee',
 'LaundryinBuilding',
 'FitnessCenter',
 'Pre-War',
 'LaundryinUnit',
 'RoofDeck',
 'OutdoorSpace',
 'DiningRoom',
 'HighSpeedInternet',
 'Balcony',
 'SwimmingPool',
 'LaundryInBuilding',
 'NewConstruction',
 'Terrace']

In [11]:
for feature in top_20_features:
    train[feature] = train["features"].str.contains(feature, regex=False).astype(int)
    test[feature] = test["features"].str.contains(feature, regex=False).astype(int)

In [12]:
features_list = top_20_features + ['bathrooms', 'bedrooms']
len(features_list)

22

In [13]:
new_train = train.drop(columns='features')
new_test = test.drop(columns='features')

lower = new_train['price'].quantile(0.01)
upper = new_train['price'].quantile(0.99)
new_test = new_test[new_test['price'].between(lower, upper) & (new_test['bedrooms'] < 5) & (new_test['bathrooms'] < 4)]
new_train = new_train[new_train['price'].between(lower, upper) & (new_train['bedrooms'] < 5) & (new_train['bathrooms'] < 4)]

# 4. Models implementation — Linear regression

## 4.1 класс линейной регрессии 
с реализованным SGD и аналитическим обучением

регулициями (задание 5)

batch-GD и minibatch-GD (задание 11) 

In [14]:
class customLinReg:
    def __init__(self, solver="sgd", lr = 0.01, epochs = 100, random_state = None, reg=None, l1_ratio = 0.5, batch_size = 32):
        self.solver = solver
        self.lr = lr
        self.epochs = epochs
        self.random_state = random_state
        self.reg = reg
        self.l1_ratio = l1_ratio
        self.batch_size = batch_size

    def __init_weights(self, features_count):
        if self.random_state != None:
            self.random_generator = np.random.RandomState(self.random_state)
        else:
            self.random_generator = np.random
        self.weights = self.random_generator.normal(loc=0.0, scale=0.01, size=1 + features_count)
    
    def __pred(self, xi):
        xi = np.array(xi, dtype=float)
        return np.dot(xi, self.weights[1:]) + self.weights[0]
    
    def __ridge(self):
        return (1 - self.l1_ratio) * 2 * self.weights[1:]
    def __lasso(self):
        return self.l1_ratio * np.sign(self.weights[1:])
    def __elastic(self):
        return self.__ridge() + self.__lasso()
    def __reg_term(self):
        if self.reg == None: return 0
        elif self.reg == 'ridge': return self.__ridge()
        elif self.reg == 'lasso': return self.__lasso()
        elif self.reg == 'elastic': return self.__elastic()
        
    def predict(self, X):
        predicts = np.dot(X, self.weights[1:]) + self.weights[0]
        return predicts

    def __analytical(self, X_train, y_train):

        X_b = np.hstack([np.ones((X_train.shape[0], 1)), X_train])

        XTX = np.dot(X_b.T, X_b)

        if self.reg is not None and self.reg == 'ridge':
            if self.reg == 'ridge':
                n_features = X_b.shape[1]
                I = np.eye(n_features)
                I[0, 0] = 0

                XTX = XTX + (1 - self.l1_ratio) * I
        self.weights = np.dot(np.dot(np.linalg.pinv(XTX), X_b.T), y_train)

        return self
    
    def __sgd(self, X_train, y_train):
        n_samples, n_features = X_train.shape
        for _ in range(self.epochs):
            indices = np.random.permutation(n_samples)
            for i in indices:
                xi = X_train[i]
                target = y_train[i]
                pred = self.__pred(xi)

                dw = (pred - target) * xi + self.__reg_term()
                db = (pred - target)

                self.weights[1:] -= self.lr * dw
                self.weights[0] -= self.lr * db

    def __batch(self, X_train, y_train):  
        n_samples, n_features = X_train.shape
        for _ in range(self.epochs):
            reg_term = self.__reg_term()
            pred = self.predict(X_train)
            dw = (1/n_samples) * np.dot(X_train.T, (pred - y_train)) + (1/n_samples) * reg_term
            db = (1/n_samples) * np.sum(pred - y_train)

            self.weights[0] -= self.lr * db
            self.weights[1:] -= self.lr * dw

    def __mini_batch(self, X_train, y_train):
        n_samples, n_features = X_train.shape

        for _ in range(self.epochs):
            indices = np.random.permutation(n_samples)

            for start in range(0, n_samples, self.batch_size):
                batch_idx = indices[start:start + self.batch_size]

                X_batch = X_train[batch_idx]
                y_batch = y_train[batch_idx]

                pred = self.predict(X_batch)

                dw = (1 / len(X_batch)) * np.dot(X_batch.T, (pred - y_batch)) + (1/self.batch_size) * self.__reg_term()
                db = (1 / len(X_batch)) * np.sum(pred - y_batch)

                self.weights[1:] -= self.lr * dw
                self.weights[0] -= self.lr * db


    def fit(self, X_train, y_train):

        X_train = np.array(X_train, dtype=float)
        y_train = np.array(y_train, dtype=float)
        self.__init_weights(X_train.shape[1])

        if self.solver == "sgd":
            self.__sgd(X_train, y_train) 
        elif self.solver == "analytical":
            self.__analytical(X_train, y_train)
        elif self.solver == "batch":
            self.__batch(X_train, y_train)
        elif self.solver == "minibatch": 
            self.__mini_batch(X_train, y_train)


## 4.2 Что такое детерменированная модель?
это модель которая при одних весах и входных данных всегда выдает один и тот же результат (то есть отсутсвует элементы случайности)

чтобы модель сделать детерменнированной нужно добавить в конструктор класса random_state, который устанавливается как сид для генератора 

## 4.3 функция для подсчета R2

In [15]:
def my_R2(y_true, y_pred):
    mean_true = np.mean(y_true)
    mean_mse = np.sum((y_true - mean_true)**2)
    pred_mse = np.sum((y_true - y_pred)**2)
    return 1 - (pred_mse/mean_mse)

## 4.4 функция для предсказаний и создания таблиц с метриками моделей

In [16]:
def get_model_table(models_dict, X_tr, X_tst, y_tr, y_tst):                
    results = []                                                           
    model_names = []                                                       
    for name, model in models_dict.items():                               
        model.fit(X_tr, y_tr)                                              
        pred_train = model.predict(X_tr)                                   
        pred_test = model.predict(X_tst)                                   
                                                                            
        model_names.append(name)                                           
        results.append({                                                   
             ('MAE', 'Train'): mean_absolute_error(y_tr, pred_train),      
             ('MAE', 'Test'): mean_absolute_error(y_tst, pred_test),       
             ('RMSE', 'Train'): root_mean_squared_error(y_tr, pred_train), 
            ('RMSE', 'Test'): root_mean_squared_error(y_tst, pred_test),   
            ('R2', 'Train'): my_R2(y_tr, pred_train),                      
            ('R2', 'Test'): my_R2(y_tst, pred_test)                        
        })                                                                 
                                                                           
    df = pd.DataFrame(results, index=model_names)                          
    df.columns = pd.MultiIndex.from_tuples(df.columns)                     
    df.index.name = 'Model'                                                
    return df                                                             


In [17]:
X_train = new_train.drop(columns='price')
y_train = new_train['price']


In [18]:
X_test = new_test.drop(columns='price')
y_test = new_test['price']

In [19]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    'Custom LinReg (analytical)': customLinReg(solver='analytical'),
    'Custom LinReg (SGD)': customLinReg(epochs=20),
    "Custom Ridge(analytical)": customLinReg(solver="analytical", reg='ridge'),
    'Custom Ridge': customLinReg(reg='ridge'),
    'Custom Lasso': customLinReg(reg='lasso'),
    'Custom ElasticNet': customLinReg(reg='elastic'),
}

## таблица с метриками

In [20]:
linregs_table = get_model_table(
    {
        'Linear Regression': LinearRegression(),
        'Custom LinReg (analytical)': customLinReg(solver='analytical'),
        'Custom LinReg (SGD)': customLinReg(epochs=20, random_state=42),
    },
    X_train,
    X_test,
    y_train,
    y_test
)
linregs_table 

MAE                     RMSE               \
                                 Train        Test        Train         Test   
Model                                                                          
Linear Regression           729.137477  732.845517  1049.893799  1052.051328   
Custom LinReg (analytical)  729.137477  732.845517  1049.893799  1052.051328   
Custom LinReg (SGD)         720.886917  724.867868  1053.285600  1055.238358   

                                  R2            
                               Train      Test  
Model                                           
Linear Regression           0.540808  0.541577  
Custom LinReg (analytical)  0.540808  0.541577  
Custom LinReg (SGD)         0.537836  0.538795

# 5. Regularized models implementation — Ridge, Lasso, ElasticNet

обучение моделей и сохранение метрик

In [21]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    'Custom LinReg (analytical)': customLinReg(solver='analytical'),
    'Custom LinReg (SGD)': customLinReg(epochs=20),
    "Custom Ridge(analytical)": customLinReg(solver="analytical", reg='ridge'),
    'Custom Ridge': customLinReg(reg='ridge'),
    'Custom Lasso': customLinReg(reg='lasso'),
    'Custom ElasticNet': customLinReg(reg='elastic'),
}

In [22]:
regulized_table = get_model_table(
    models,
    X_train, 
    X_test,
    y_train,
    y_test
)
regulized_table

MAE                     RMSE               \
                                 Train        Test        Train         Test   
Model                                                                          
Linear Regression           729.137477  732.845517  1049.893799  1052.051328   
Ridge                       729.135472  732.843732  1049.893803  1052.051469   
Lasso                       728.896775  732.742846  1049.938632  1052.104741   
ElasticNet                  811.829824  816.794522  1191.214767  1195.370206   
Custom LinReg (analytical)  729.137477  732.845517  1049.893799  1052.051328   
Custom LinReg (SGD)         724.760596  728.447396  1075.567577  1076.806080   
Custom Ridge(analytical)    729.136474  732.844625  1049.893800  1052.051398   
Custom Ridge                856.452624  861.034417  1289.236803  1293.701626   
Custom Lasso                780.599542  783.856235  1075.067348  1077.749621   
Custom ElasticNet           850.133098  855.297500  1258.305164  1262.837921   

                                  R2            
                               Train      Test  
Model                                           
Linear Regression           0.540808  0.541577  
Ridge                       0.540808  0.541577  
Lasso                       0.540769  0.541530  
ElasticNet                  0.408869  0.408169  
Custom LinReg (analytical)  0.540808  0.541577  
Custom LinReg (SGD)         0.518076  0.519750  
Custom Ridge(analytical)    0.540808  0.541577  
Custom Ridge                0.307581  0.306796  
Custom Lasso                0.518524  0.518908  
Custom ElasticNet           0.340408  0.339477

# 6. Feature normalization

### 6.1 Почему нормализация обязательна?
Нормализация обязательна для моделей, использующих метрики расстояния (например, KNN) или градиентную оптимизацию (например, SGD). Если признаки имеют разный масштаб, градиентный спуск будет сходиться гораздо дольше, так как поверхность функции потерь будет сильно вытянутой. Признаки с большим масштабом будут доминировать при обновлении весов. Нормализация не является обязательной для моделей на основе деревьев.

## 6.2 формулы minmax и standartization

minmax

$x_{i \ scaled} = {\frac{x_i - x_{min}}{x_{max} - x_{min}}}$

standartization

$x_{i \ scaled} = {\frac{x_i - \mu}{\sigma}}$

$\mu - mean$

$\sigma - standart\ deviation$

## 6.3 Классы my_MinMaxScaler и my_StandartScaler

In [23]:
class my_MinMaxScaler:
    def __init__(self):
        self.min_ = None
        self.max_ = None

    def fit(self, X):
        X = np.array(X)
        self.min_ = X.min(axis=0)
        self.max_ = X.max(axis=0)
        return self
    
    def transform(self, X):
        X = np.asarray(X)
        range_ = self.max_ - self.min_
        range_ = np.where(range_ == 0, 1, range_)
        X_scaled = (X - self.min_) / (range_)
        X_scaled[np.isnan(X_scaled)] = 0  
        return X_scaled

    def fit_transform(self, X):
        return self.fit(X).transform(X)


In [24]:
class my_StandartScaler:
    def __init__(self):
        self.mean_ = None
        self.std_ = None
    
    def fit(self, X):
        X = np.array(X)

        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        return self
    
    def transform(self, X):
        X = np.asarray(X)
        self.std_ = np.where(self.std_ == 0, 1, self.std_)

        X_scaled = (X - self.mean_) / self.std_
        return X_scaled
       
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)


In [25]:
mm_scaler = my_MinMaxScaler()
std_scaler = my_StandartScaler()

## 6.4 инициализация MinMaxScaler и StandartScaler из sklearn

In [26]:
sk_mm_scaler = MinMaxScaler()
sk_std_scaler = StandardScaler()

In [27]:
X_train

,bathrooms,bedrooms,Elevator,CatsAllowed,HardwoodFloors,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,...,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
4,1.0,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
6,1.0,2,1,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
9,1.0,2,1,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
10,1.5,3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
15,1.0,0,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124000,1.0,3,1,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
124002,1.0,2,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
124004,1.0,1,1,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
124008,1.0,2,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


нормализация данных

In [28]:
X_train_MM = mm_scaler.fit_transform(X_train.iloc[:, :2])
X_test_MM = mm_scaler.transform(X_test.iloc[:, :2])

In [29]:
X_train_MM = np.hstack((X_train_MM, X_train.iloc[:, 2:]))
X_test_MM = np.hstack((X_test_MM, X_test.iloc[:, 2:]))

In [30]:
X_train_MMsk = sk_mm_scaler.fit_transform(X_train.iloc[:, :2])
X_test_MMsk = sk_mm_scaler.transform(X_test.iloc[:, :2])

In [31]:
X_train_MMsk = np.hstack((X_train_MMsk, X_train.iloc[:, 2:]))
X_test_MMsk = np.hstack((X_test_MMsk, X_test.iloc[:, 2:]))

In [32]:
X_train_std = std_scaler.fit_transform(X_train.iloc[:, :2])
X_test_std = std_scaler.transform(X_test.iloc[:, :2])

In [33]:
X_train_std = np.hstack((X_train_std, X_train.iloc[:, 2:]))
X_test_std = np.hstack((X_test_std, X_test.iloc[:, 2:]))

In [34]:
X_train_stdsk = sk_std_scaler.fit_transform(X_train.iloc[:, :2])
X_test_stdsk = sk_std_scaler.transform(X_test.iloc[:, :2])

In [35]:
X_train_stdsk = np.hstack((X_train_stdsk, X_train.iloc[:, 2:]))
X_test_stdsk = np.hstack((X_test_stdsk, X_test.iloc[:, 2:]))

## 6.5 сравнение результатов


In [36]:
np.allclose(X_train_std, X_train_stdsk)

True

In [37]:
np.allclose(X_train_MM, X_train_MMsk)

True

# 7. Fit custom and sklearn models with normalized data

In [38]:
minmax_table = get_model_table(models,
        X_train_MM,
        X_test_MM,
        y_train,
        y_test
        )
minmax_table

MAE                      RMSE  \
                                  Train         Test        Train   
Model                                                               
Linear Regression            729.137477   732.845517  1049.893799   
Ridge                        729.131739   732.840107  1049.894363   
Lasso                        728.930837   732.786751  1049.965490   
ElasticNet                  1006.061715  1009.455832  1420.027249   
Custom LinReg (analytical)   729.137477   732.845517  1049.893799   
Custom LinReg (SGD)          724.729662   728.799596  1052.240079   
Custom Ridge(analytical)     729.134577   732.842738  1049.893940   
Custom Ridge                1053.562571  1057.040214  1476.871431   
Custom Lasso                 745.533277   749.327873  1053.687280   
Custom ElasticNet           1108.817305  1111.854389  1463.114075   

                                               R2            
                                   Test     Train      Test  
Model                                                        
Linear Regression           1052.051328  0.540808  0.541577  
Ridge                       1052.053513  0.540808  0.541575  
Lasso                       1052.137050  0.540745  0.541502  
ElasticNet                  1424.044841  0.159966  0.160076  
Custom LinReg (analytical)  1052.051328  0.540808  0.541577  
Custom LinReg (SGD)         1054.061163  0.538753  0.539824  
Custom Ridge(analytical)    1052.052281  0.540808  0.541576  
Custom Ridge                1481.112383  0.091366  0.091409  
Custom Lasso                1056.267052  0.537484  0.537896  
Custom ElasticNet           1467.044753  0.108216  0.108586

In [39]:
standart_table = get_model_table(models,
        X_train_std,
        X_test_std,
        y_train,
        y_test
        )
standart_table

MAE                     RMSE               \
                                 Train        Test        Train         Test   
Model                                                                          
Linear Regression           729.137477  732.845517  1049.893799  1052.051328   
Ridge                       729.136016  732.844329  1049.893799  1052.051336   
Lasso                       728.908371  732.756639  1049.937475  1052.100922   
ElasticNet                  764.263476  768.875202  1102.526556  1105.949043   
Custom LinReg (analytical)  729.137477  732.845517  1049.893799  1052.051328   
Custom LinReg (SGD)         783.242955  785.238590  1109.669116  1110.778037   
Custom Ridge(analytical)    729.136745  732.844922  1049.893799  1052.051332   
Custom Ridge                780.193244  785.098077  1146.563605  1150.143822   
Custom Lasso                728.533083  732.844082  1072.511547  1074.863513   
Custom ElasticNet           877.536103  882.895041  1245.498025  1249.894184   

                                  R2            
                               Train      Test  
Model                                           
Linear Regression           0.540808  0.541577  
Ridge                       0.540808  0.541577  
Lasso                       0.540770  0.541534  
ElasticNet                  0.493614  0.493403  
Custom LinReg (analytical)  0.540808  0.541577  
Custom LinReg (SGD)         0.487032  0.488969  
Custom Ridge(analytical)    0.540808  0.541577  
Custom Ridge                0.452354  0.452106  
Custom Lasso                0.520810  0.521481  
Custom ElasticNet           0.353766  0.352948

# 8. Overfit models

## 8.2 создание полиномиальных признаков

In [40]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    'Custom LinReg (analytical)': customLinReg(solver='analytical'),
    "Custom Ridge(analytical)": customLinReg(solver="analytical", reg='ridge'),
    # 'Custom Lasso': customLinReg(reg='lasso'),
    # 'Custom ElasticNet': customLinReg(reg='elastic'),
}

In [41]:
pf = PolynomialFeatures(10)
X_train_poly = pf.fit_transform(X_train.iloc[:, :2])
X_test_poly = pf.transform(X_test.iloc[:, :2])

In [42]:
X_train_poly = np.hstack((X_train_poly, X_train.iloc[:, 2:]))
X_test_poly = np.hstack((X_test_poly, X_test.iloc[:, 2:]))

In [43]:
poly_table = get_model_table(
    models,
    X_train_poly, 
    X_test_poly,
    y_train,
    y_test
)

/opt/homebrew/Caskroom/miniconda/base/envs/.venv/lib/python3.14/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=7.94715e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/opt/homebrew/Caskroom/miniconda/base/envs/.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.473e+10, tolerance: 1.154e+07
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Caskroom/miniconda/base/envs/.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.696e+10, tolerance: 1.154e+07
  model = cd_fast.enet_coordinate_d

In [44]:
poly_table

MAE                     RMSE               \
                                 Train        Test        Train         Test   
Model                                                                          
Linear Regression           699.285552  736.308881  1007.056955  3827.132702   
Ridge                       699.274681  733.993705  1007.058704  3584.773349   
Lasso                       700.825279  704.519110  1011.102602  1019.204876   
ElasticNet                  721.215305  725.624045  1038.268343  1046.654641   
Custom LinReg (analytical)  699.285616  736.301845  1007.056955  3826.383705   
Custom Ridge(analytical)    699.277354  735.011150  1007.057448  3691.381049   

                                  R2            
                               Train      Test  
Model                                           
Linear Regression           0.577515 -5.066521  
Ridge                       0.577513 -4.322505  
Lasso                       0.574113  0.569755  
ElasticNet                  0.550921  0.546268  
Custom LinReg (analytical)  0.577515 -5.064146  
Custom Ridge(analytical)    0.577514 -4.643784

# 9. Native models

In [45]:
native_models = {
    "naive mean": DummyRegressor(strategy='mean'),
    "naive median": DummyRegressor(strategy='median')
}

In [46]:
native_table = get_model_table(native_models, X_train, X_test,
      y_train, y_test)
native_table

MAE                      RMSE                     R2  \
                    Train         Test        Train         Test     Train   
Model                                                                        
naive mean    1109.850474  1113.549434  1549.345458  1553.831176  0.000000   
naive median  1061.961770  1065.860541  1591.918209  1595.988377 -0.055711   

                            
                      Test  
Model                       
naive mean   -6.909514e-07  
naive median -5.499909e-02

# 10. Compare results

In [47]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    'Custom LinReg (SGD)': customLinReg(epochs=20),
    'Custom Ridge': customLinReg(reg='ridge'),
    'Custom Lasso': customLinReg(reg='lasso'),
    'Custom ElasticNet': customLinReg(reg='elastic'),
}

results_df = get_model_table(models, X_train, X_test, y_train, y_test)
results_df

MAE                     RMSE               \
                          Train        Test        Train         Test   
Model                                                                   
Linear Regression    729.137477  732.845517  1049.893799  1052.051328   
Ridge                729.135472  732.843732  1049.893803  1052.051469   
Lasso                728.896775  732.742846  1049.938632  1052.104741   
ElasticNet           811.829824  816.794522  1191.214767  1195.370206   
Custom LinReg (SGD)  722.592686  726.759508  1053.602083  1055.845918   
Custom Ridge         858.529428  863.653138  1292.210779  1296.346425   
Custom Lasso         728.924383  733.331968  1056.589760  1058.594611   
Custom ElasticNet    895.776439  900.959160  1257.619717  1262.409883   

                           R2            
                        Train      Test  
Model                                    
Linear Regression    0.540808  0.541577  
Ridge                0.540808  0.541577  
Lasso                0.540769  0.541530  
ElasticNet           0.408869  0.408169  
Custom LinReg (SGD)  0.537559  0.538264  
Custom Ridge         0.304383  0.303959  
Custom Lasso         0.534932  0.535857  
Custom ElasticNet    0.341126  0.339925

In [48]:
minmax_table

MAE                      RMSE  \
                                  Train         Test        Train   
Model                                                               
Linear Regression            729.137477   732.845517  1049.893799   
Ridge                        729.131739   732.840107  1049.894363   
Lasso                        728.930837   732.786751  1049.965490   
ElasticNet                  1006.061715  1009.455832  1420.027249   
Custom LinReg (analytical)   729.137477   732.845517  1049.893799   
Custom LinReg (SGD)          724.729662   728.799596  1052.240079   
Custom Ridge(analytical)     729.134577   732.842738  1049.893940   
Custom Ridge                1053.562571  1057.040214  1476.871431   
Custom Lasso                 745.533277   749.327873  1053.687280   
Custom ElasticNet           1108.817305  1111.854389  1463.114075   

                                               R2            
                                   Test     Train      Test  
Model                                                        
Linear Regression           1052.051328  0.540808  0.541577  
Ridge                       1052.053513  0.540808  0.541575  
Lasso                       1052.137050  0.540745  0.541502  
ElasticNet                  1424.044841  0.159966  0.160076  
Custom LinReg (analytical)  1052.051328  0.540808  0.541577  
Custom LinReg (SGD)         1054.061163  0.538753  0.539824  
Custom Ridge(analytical)    1052.052281  0.540808  0.541576  
Custom Ridge                1481.112383  0.091366  0.091409  
Custom Lasso                1056.267052  0.537484  0.537896  
Custom ElasticNet           1467.044753  0.108216  0.108586

In [49]:
standart_table

MAE                     RMSE               \
                                 Train        Test        Train         Test   
Model                                                                          
Linear Regression           729.137477  732.845517  1049.893799  1052.051328   
Ridge                       729.136016  732.844329  1049.893799  1052.051336   
Lasso                       728.908371  732.756639  1049.937475  1052.100922   
ElasticNet                  764.263476  768.875202  1102.526556  1105.949043   
Custom LinReg (analytical)  729.137477  732.845517  1049.893799  1052.051328   
Custom LinReg (SGD)         783.242955  785.238590  1109.669116  1110.778037   
Custom Ridge(analytical)    729.136745  732.844922  1049.893799  1052.051332   
Custom Ridge                780.193244  785.098077  1146.563605  1150.143822   
Custom Lasso                728.533083  732.844082  1072.511547  1074.863513   
Custom ElasticNet           877.536103  882.895041  1245.498025  1249.894184   

                                  R2            
                               Train      Test  
Model                                           
Linear Regression           0.540808  0.541577  
Ridge                       0.540808  0.541577  
Lasso                       0.540770  0.541534  
ElasticNet                  0.493614  0.493403  
Custom LinReg (analytical)  0.540808  0.541577  
Custom LinReg (SGD)         0.487032  0.488969  
Custom Ridge(analytical)    0.540808  0.541577  
Custom Ridge                0.452354  0.452106  
Custom Lasso                0.520810  0.521481  
Custom ElasticNet           0.353766  0.352948

In [50]:
poly_table

MAE                     RMSE               \
                                 Train        Test        Train         Test   
Model                                                                          
Linear Regression           699.285552  736.308881  1007.056955  3827.132702   
Ridge                       699.274681  733.993705  1007.058704  3584.773349   
Lasso                       700.825279  704.519110  1011.102602  1019.204876   
ElasticNet                  721.215305  725.624045  1038.268343  1046.654641   
Custom LinReg (analytical)  699.285616  736.301845  1007.056955  3826.383705   
Custom Ridge(analytical)    699.277354  735.011150  1007.057448  3691.381049   

                                  R2            
                               Train      Test  
Model                                           
Linear Regression           0.577515 -5.066521  
Ridge                       0.577513 -4.322505  
Lasso                       0.574113  0.569755  
ElasticNet                  0.550921  0.546268  
Custom LinReg (analytical)  0.577515 -5.064146  
Custom Ridge(analytical)    0.577514 -4.643784

In [51]:
native_table

MAE                      RMSE                     R2  \
                    Train         Test        Train         Test     Train   
Model                                                                        
naive mean    1109.850474  1113.549434  1549.345458  1553.831176  0.000000   
naive median  1061.961770  1065.860541  1591.918209  1595.988377 -0.055711   

                            
                      Test  
Model                       
naive mean   -6.909514e-07  
naive median -5.499909e-02


**Самая стабильная модель:** Ridge-регрессия в целом более стабильна, так как она предотвращает чрезмерный рост весов, даже при наличии коррелированных признаков.

# 11. Addition task

## 11.1 logarithmic functions


In [52]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

def get_model_table_log(models_dict, X_tr, X_tst, y_tr_log, y_tst_log, y_tr_orig, y_tst_orig):
    results = []
    model_names = []
    for name, model in models_dict.items():
        model.fit(X_tr, y_tr_log)

        pred_train = np.expm1(model.predict(X_tr))
        pred_test = np.expm1(model.predict(X_tst))
                                                                            
        model_names.append(name)                                           
        results.append({                                                   
             ('MAE', 'Train'): mean_absolute_error(y_tr_orig, pred_train),      
             ('MAE', 'Test'): mean_absolute_error(y_tst_orig, pred_test),       
             ('RMSE', 'Train'): root_mean_squared_error(y_tr_orig, pred_train), 
            ('RMSE', 'Test'): root_mean_squared_error(y_tst_orig, pred_test),   
            ('R2', 'Train'): my_R2(y_tr_orig, pred_train),                      
            ('R2', 'Test'): my_R2(y_tst_orig, pred_test)                        
        })                                                                 
                                                                           
    df = pd.DataFrame(results, index=model_names)                          
    df.columns = pd.MultiIndex.from_tuples(df.columns)                     
    df.index.name = 'Model'                                                
    return df

log_models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    'Custom LinReg (analytical)': customLinReg(solver='analytical'),
    'Custom LinReg (SGD)': customLinReg(epochs=20),
    "Custom Ridge(analytical)": customLinReg(solver="analytical", reg='ridge'),
    'Custom Ridge': customLinReg(reg='ridge'),
    'Custom Lasso': customLinReg(reg='lasso'),
    'Custom ElasticNet': customLinReg(reg='elastic'),
}

log_data_table = get_model_table_log(log_models, X_train_std, X_test_std, y_train_log, y_test_log, y_train, y_test)
log_data_table

MAE                      RMSE  \
                                  Train         Test        Train   
Model                                                               
Linear Regression            717.398633   721.172849  1053.823204   
Ridge                        717.396925   721.171256  1053.824184   
Lasso                       1066.301743  1070.014573  1571.365774   
ElasticNet                  1066.301743  1070.014573  1571.365774   
Custom LinReg (analytical)   717.398633   721.172849  1053.823204   
Custom LinReg (SGD)          736.714252   740.984557  1080.377248   
Custom Ridge(analytical)     717.397779   721.172053  1053.823694   
Custom Ridge                 799.647259   804.844334  1200.461981   
Custom Lasso                1062.050030  1065.823117  1558.038352   
Custom ElasticNet           1085.295620  1089.127736  1553.637211   

                                               R2            
                                   Test     Train      Test  
Model                                                        
Linear Regression           1055.708416  0.537364  0.538384  
Ridge                       1055.709354  0.537364  0.538383  
Lasso                       1575.573928 -0.028627 -0.028183  
ElasticNet                  1575.573928 -0.028627 -0.028183  
Custom LinReg (analytical)  1055.708416  0.537364  0.538384  
Custom LinReg (SGD)         1082.626235  0.513756  0.514544  
Custom Ridge(analytical)    1055.708885  0.537364  0.538384  
Custom Ridge                1203.481948  0.399656  0.400110  
Custom Lasso                1562.300407 -0.011253 -0.010932  
Custom ElasticNet           1558.024545 -0.005548 -0.005405

## 11.2 deleting outliners

**Почему мы удаляем выбросы только из обучающей выборки?**

1. **Защита обучения**: Линейная регрессия минимизирует квадрат ошибки (MSE). Один аномальный объект с ценой в 10 раз выше рыночной может развернуть плоскость регрессии так, что она станет неэффективной для 99% нормальных квартир. Удаляя выбросы из обучения, мы получаем более устойчивую и адекватную модель.
2. **Честность оценки**: Тестовая выборка должна представлять реальный мир. В реальности аномалии случаются, и модель должна уметь с ними сталкиваться. Если мы «почистим» тест, наши метрики будут искусственно завышены. Мы должны видеть, как модель ведет себя на реальных, порой «грязных» данных.
